##### Purpose

Run and compare multiple Telco Churn neural-network configurations using MLflow.

The notebook evaluates how changes in neural-network architecture and training hyperparameters affect validation performance.

The goal is to identify the strongest candidate configuration without using the test dataset for model selection.

``` text

Training Data
      ↓
Controlled Experiments
      ↓
┌───────────┬───────────┬───────────┐
│ Run A     │ Run B     │ Run C     │
│ 32 → 16   │ 64 → 32   │ 32 → 16   │
│ LR .001   │ LR .001   │ LR .0005  │
└───────────┴───────────┴───────────┘
      ↓
MLflow Tracking
      ↓
Compare Validation Metrics
      ↓
Select Best Candidate
      ↓
08_model_registration

```

##### Production Design

The responsibilities are deliberately separated:

``` text

src/
├── preprocessing.py
├── model.py
└── training.py
        ↑
        │
07_mlflow_experiment_comparison
        │
        ↓
MLflow

```

Notebook 07 acts primarily as an experiment orchestrator.

Reusable implementation belongs in src/.

The notebook should:

``` text

Define experiment configurations
        ↓
Prepare data once
        ↓
Create a fresh model per experiment
        ↓
Train independently
        ↓
Apply early stopping
        ↓
Restore best checkpoint
        ↓
Log run to MLflow
        ↓
Collect validation results
        ↓
Compare runs
        ↓
Select candidate

```

##### Technologies Used

- Databricks
- Python
- PyTorch
- MLflow
- Pandas
- Matplotlib

##### Input

The notebook uses:

- Gold Telco Delta table
- Project configuration
- Reusable preprocessing logic
- Reusable neural-network architecture
- Reusable training logic

It uses only:

- Training set
- Validation set

for experiment selection.

The test set is intentionally not evaluated.


##### Output

The notebook produces:

Multiple MLflow runs

- Run parameters
- Training history
- Best validation loss
- Best epoch
- Epochs completed
- Comparison table
- Comparison visualization
- Selected candidate Run ID
- Selected candidate configuration

##### 1. Load Project Configuration

In [0]:
%run ./00_project_config

##### 2. Import Required Components

In [0]:
import copy

import matplotlib.pyplot as plt
import mlflow
import mlflow.pytorch
import pandas as pd
import torch
import torch.nn as nn

from src.data import prepare_training_data
from src.model import TelcoChurnNN
from src.training import (
    train_one_epoch,
    validate_one_epoch,
)

src/preprocessing.py
- build_preprocessor()
- validate_feature_columns()

src/data.py
- prepare_training_data()

src/model.py
- TelcoChurnNN

src/training.py
- train_one_epoch()
- validate_one_epoch()

##### 3. Prepare Training Data Once

In [0]:
(
    train_loader,
    val_loader,
    test_loader,
    input_size,
    preprocessor,
) = prepare_training_data(
    spark=spark,
    gold_table=GOLD_TABLE,
    target_column=TARGET_COLUMN,
    columns_to_drop=COLUMNS_TO_DROP,
    numerical_features=NUMERICAL_FEATURES,
    categorical_features=CATEGORICAL_FEATURES,
    binary_features=BINARY_FEATURES,
    test_size=TEST_SIZE,
    validation_test_size=VALIDATION_TEST_SIZE,
    random_state=RANDOM_STATE,
    batch_size=BATCH_SIZE,
)

In [0]:
print(
    "Input size:",
    input_size,
)

print(
    "Training batches:",
    len(train_loader),
)

print(
    "Validation batches:",
    len(val_loader),
)

##### 4. Define Controlled Experiments

In [0]:
#Let's start with three experiments
experiment_configs = [
    {
        "experiment_name": "baseline",
        "hidden_size_1": 32,
        "hidden_size_2": 16,
        "learning_rate": 0.001,
    },
    {
        "experiment_name": "larger_network",
        "hidden_size_1": 64,
        "hidden_size_2": 32,
        "learning_rate": 0.001,
    },
    {
        "experiment_name": "lower_learning_rate",
        "hidden_size_1": 32,
        "hidden_size_2": 16,
        "learning_rate": 0.0005,
    },
]

##### 5. Define MLflow Experiment

In [0]:
EXPERIMENT_PATH = ("/Users/sujathakrishna2811@gmail.com/telco_churn_nn_experiments")

mlflow.set_experiment(
    EXPERIMENT_PATH
)

This gives us:

``` text

telco_churn_nn_experiments
        │
        ├── baseline
        ├── larger_network
        └── lower_learning_rate

```
Each execution becomes its own MLflow run.

##### 6. Create a Reusable Training Function

In [0]:
#This function performs one complete experiment.
def train_experiment(
    config,
    train_loader,
    val_loader,
    input_size,
):
    """
    Train one neural-network experiment.

    Returns:
        model
        training_history
        best_epoch
        best_validation_loss
        epochs_completed
    """

    model = TelcoChurnNN(
        input_size=input_size,
        hidden_size_1=config[
            "hidden_size_1"
        ],
        hidden_size_2=config[
            "hidden_size_2"
        ],
    )

    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config[
            "learning_rate"
        ],
    )

    best_validation_loss = float(
        "inf"
    )

    best_model_state = None

    best_epoch = 0

    epochs_without_improvement = 0

    training_history = []

    for epoch in range(
        NUM_EPOCHS
    ):

        # -------------------------
        # Training Phase
        # -------------------------

        model.train()

        total_train_loss = 0.0

        for X_batch, y_batch in train_loader:

            optimizer.zero_grad()

            logits = model(
                X_batch
            )
         
            loss = criterion(
                logits,
                y_batch,
            )

            loss.backward()

            optimizer.step()

            total_train_loss += (
                loss.item()
            )

        average_train_loss = (
            total_train_loss
            / len(train_loader)
        )

        # -------------------------
        # Validation Phase
        # -------------------------

        model.eval()

        total_validation_loss = 0.0

        with torch.no_grad():

            for X_batch, y_batch in val_loader:

                logits = model(
                    X_batch
                )
             
                validation_loss = criterion(
                    logits,
                    y_batch,
                )

                total_validation_loss += (
                    validation_loss.item()
                )

        average_validation_loss = (
            total_validation_loss
            / len(val_loader)
        )

        # -------------------------
        # Track Epoch
        # -------------------------

        training_history.append(
            {
                "epoch": epoch + 1,
                "train_loss":
                    average_train_loss,
                "validation_loss":
                    average_validation_loss,
            }
        )

        # -------------------------
        # Best Model Checkpoint
        # -------------------------

        if (
            average_validation_loss
            <
            best_validation_loss
            - MIN_DELTA
        ):

            best_validation_loss = (
                average_validation_loss
            )

            best_epoch = epoch + 1

            best_model_state = copy.deepcopy(
                model.state_dict()
            )

            epochs_without_improvement = 0

        else:

            epochs_without_improvement += 1

        # -------------------------
        # Early Stopping
        # -------------------------

        if (
            epochs_without_improvement
            >= PATIENCE
        ):

            break

    if best_model_state is None:
        raise RuntimeError(
            "Training completed without "
            "capturing a best model state."
        )

    model.load_state_dict(
        best_model_state
    )

    epochs_completed = len(
        training_history
    )

    return (
        model,
        training_history,
        best_epoch,
        best_validation_loss,
        epochs_completed,
    )

##### 7. Run MLflow Experiments

In [0]:
experiment_results = []

for config in experiment_configs:

    print(
        "\nRunning experiment:",
        config["experiment_name"],
    )

    with mlflow.start_run(
        run_name=config[
            "experiment_name"
        ]
    ) as run:

        (
            model,
            training_history,
            best_epoch,
            best_validation_loss,
            epochs_completed,
        ) = train_experiment(
            config=config,
            train_loader=train_loader,
            val_loader=val_loader,
            input_size=input_size,
        )

        # -------------------------
        # Log Parameters
        # -------------------------

        mlflow.log_params(
            {
                "hidden_size_1":
                    config[
                        "hidden_size_1"
                    ],
                "hidden_size_2":
                    config[
                        "hidden_size_2"
                    ],
                "learning_rate":
                    config[
                        "learning_rate"
                    ],
                "batch_size":
                    BATCH_SIZE,
                "configured_epochs":
                    NUM_EPOCHS,
                "patience":
                    PATIENCE,
                "min_delta":
                    MIN_DELTA,
                "input_size":
                    input_size,
            }
        )

        # -------------------------
        # Log Epoch Metrics
        # -------------------------

        for record in training_history:

            mlflow.log_metric(
                "train_loss",
                record[
                    "train_loss"
                ],
                step=record[
                    "epoch"
                ],
            )

            mlflow.log_metric(
                "validation_loss",
                record[
                    "validation_loss"
                ],
                step=record[
                    "epoch"
                ],
            )

        # -------------------------
        # Log Summary Metrics
        # -------------------------

        mlflow.log_metrics(
            {
                "best_validation_loss":
                    best_validation_loss,
                "best_epoch":
                    best_epoch,
                "epochs_completed":
                    epochs_completed,
            }
        )

        run_id = run.info.run_id

        experiment_results.append(
            {
                "experiment_name":
                    config[
                        "experiment_name"
                    ],
                "run_id":
                    run_id,
                "hidden_size_1":
                    config[
                        "hidden_size_1"
                    ],
                "hidden_size_2":
                    config[
                        "hidden_size_2"
                    ],
                "learning_rate":
                    config[
                        "learning_rate"
                    ],
                "best_epoch":
                    best_epoch,
                "epochs_completed":
                    epochs_completed,
                "best_validation_loss":
                    best_validation_loss,
            }
        )

        print(
            "Run ID:",
            run_id,
        )

        print(
            "Best epoch:",
            best_epoch,
        )

        print(
            "Best validation loss:",
            round(
                best_validation_loss,
                4,
            ),
        )

Track experiments deliberately; don't persist artifacts merely because MLflow allows it.

##### 8. Build Experiment Comparison Table

In [0]:
results_df = pd.DataFrame(
    experiment_results
)

display(
    results_df
)

##### 9. Rank Experiments

In [0]:
# For this experiment, our primary selection metric is: Best Validation Loss. Lower is better.

ranked_results_df = (
    results_df
    .sort_values(
        by="best_validation_loss",
        ascending=True,
    )
    .reset_index(
        drop=True
    )
)

display(
    ranked_results_df
)

##### 10. Identify Best Candidate

In [0]:
best_candidate = (
    ranked_results_df.iloc[0]
)

print(
    "Selected experiment:",
    best_candidate[
        "experiment_name"
    ],
)

print(
    "Run ID:",
    best_candidate[
        "run_id"
    ],
)

print(
    "Best validation loss:",
    round(
        best_candidate[
            "best_validation_loss"
        ],
        4,
    ),
)

print(
    "Best epoch:",
    int(
        best_candidate[
            "best_epoch"
        ]
    ),
)

##### 11. Display Selected Architecture

In [0]:
print(
    "Hidden Layer 1:",
    int(
        best_candidate[
            "hidden_size_1"
        ]
    ),
)

print(
    "Hidden Layer 2:",
    int(
        best_candidate[
            "hidden_size_2"
        ]
    ),
)

print(
    "Learning Rate:",
    best_candidate[
        "learning_rate"
    ],
)

WHAT configuration won and WHICH MLflow run produced it

##### 12. Visualize Validation-Loss Comparison

In [0]:
plot_df = (
    ranked_results_df[
        [
            "experiment_name",
            "best_validation_loss",
        ]
    ]
    .set_index(
        "experiment_name"
    )
)

ax = plot_df.plot(
    kind="bar",
    figsize=(8, 5),
    legend=False,
)

ax.set_title(
    "Neural Network Experiment Comparison"
)

ax.set_xlabel(
    "Experiment"
)

ax.set_ylabel(
    "Best Validation Loss"
)

plt.xticks(
    rotation=0
)

plt.grid(
    axis="y"
)

plt.show()

The shortest bar represents the lowest validation loss.

##### 13. Compare Training Efficiency

Validation performance isn't only useful observation.

In [0]:
efficiency_df = ranked_results_df[
    [
        "experiment_name",
        "best_epoch",
        "epochs_completed",
        "best_validation_loss",
    ]
]

display(
    efficiency_df
)

Which model achieved the best result?

How many epochs did it need?

When did early stopping occur?

Did a larger architecture actually help?

##### 14. Retrieve Runs Directly from MLflow

In [0]:
experiment = mlflow.get_experiment_by_name(
    EXPERIMENT_PATH
)

runs_df = mlflow.search_runs(
    experiment_ids=[
        experiment.experiment_id
    ],
    order_by=[
        "metrics.best_validation_loss ASC"
    ],
)

display(
    runs_df[
        [
            "run_id",
            "tags.mlflow.runName",
            "params.hidden_size_1",
            "params.hidden_size_2",
            "params.learning_rate",
            "metrics.best_validation_loss",
            "metrics.best_epoch",
            "metrics.epochs_completed",
        ]
    ]
)

Notebook memory
     ✗

MLflow Tracking Store
     ✓

##### 15. Understand Logging vs Tracking vs Comparison


``` text 

LOGGING
   ↓
Record information from one run

Example:
learning_rate = .001
best_val_loss = .418


TRACKING
   ↓
Persist many runs in MLflow


COMPARISON
   ↓
Compare those runs


MODEL SELECTION
   ↓
Choose candidate using
validation criteria


REGISTRATION
   ↓
Promote controlled model artifact
for lifecycle management

```

##### 16. Why We Do Not Use Test Data Here


``` text

Training Data
     ↓
Learn weights


Validation Data
     ↓
Architecture selection
Learning-rate selection
Early stopping
Experiment comparison


Test Data
     ↓
Final unbiased evaluation

```

##### 17. Production Selection Is More Than One Number


Suppose:

``` text

Model A
Validation Loss = 0.4179
Parameters = 2,017

Model B
Validation Loss = 0.4178
Parameters = 5,057

```
Technically: Model B wins by 0.0001.

But production engineering might still choose Model A because:

``` text

almost identical performance
+
smaller model
+
simpler architecture
+
lower inference cost

```

So production selection considers:

``` text

Predictive performance
+
Stability
+
Complexity
+
Latency
+
Compute cost
+
Maintainability
+
Business requirements

```

MLflow provides evidence for the decision.

It does not make the business decision for us.

##### 18. Experiment Summary

In [0]:
experiment_summary = {
    "number_of_experiments":
        len(results_df),

    "selected_experiment":
        best_candidate[
            "experiment_name"
        ],

    "selected_run_id":
        best_candidate[
            "run_id"
        ],

    "best_validation_loss":
        float(
            best_candidate[
                "best_validation_loss"
            ]
        ),

    "best_epoch":
        int(
            best_candidate[
                "best_epoch"
            ]
        ),

    "hidden_size_1":
        int(
            best_candidate[
                "hidden_size_1"
            ]
        ),

    "hidden_size_2":
        int(
            best_candidate[
                "hidden_size_2"
            ]
        ),

    "learning_rate":
        float(
            best_candidate[
                "learning_rate"
            ]
        ),
}

In [0]:
for key, value in experiment_summary.items():

    print(
        f"{key}: {value}"
    )

##### Key Learnings

1. MLflow experiments allow multiple model-training runs to be tracked and compared systematically.

2. Each MLflow run represents one specific combination of model architecture and training hyperparameters.

3. Controlled experiments help identify which change caused a performance difference.

4. A fresh neural-network model must be created for every experiment.

5. Reusing an already-trained model would invalidate the comparison because later experiments would start from learned weights.

6. Training data is used to learn model parameters.

7. Validation data is used for early stopping, hyperparameter comparison, and candidate selection.

8. Test data should not be repeatedly used during experiment selection.

9. Training loss and validation loss should be logged for each epoch so model-learning behavior can be inspected.

10. Best validation loss summarizes the strongest validation checkpoint within a run.

11. Early stopping reduces unnecessary training when validation performance stops improving.

12. MLflow parameters describe how a run was configured.

13. MLflow metrics describe how that run performed.

14. MLflow Run IDs provide reproducible identities for individual experiments.

15. MLflow search allows experiment results to be recovered without depending on notebook memory.

16. Logging records one experiment.

17. Tracking organizes multiple experiments.

18. Comparison evaluates those experiments.

19. Model selection identifies the candidate that should proceed in the lifecycle.

20. The numerically best model is not automatically the best production model.

21. Performance, model size, training cost, inference cost, interpretability, and maintainability should all be considered.

22. Experiment tracking becomes increasingly important as deep-learning architectures and hyperparameter combinations become more complex.

23. MLflow provides the engineering lifecycle around neural-network experimentation; PyTorch provides the model-training mechanics.

##### Conclusion

This notebook used controlled neural-network experiments to demonstrate production-style MLflow experiment tracking and model comparison.

Three configurations were evaluated:

1. Baseline neural network
   32 → 16
   learning rate = 0.001

2. Larger neural network
   64 → 32
   learning rate = 0.001

3. Baseline architecture with a lower learning rate
   32 → 16
   learning rate = 0.0005

Each experiment:

- created a fresh neural network,
- trained using the same training data,
- evaluated using the same validation data,
- applied best-model checkpointing,
- applied early stopping,
- logged parameters and metrics to MLflow,
- and produced an independently identifiable MLflow run.

The experiments were ranked using best validation loss.

The test dataset was intentionally excluded from experiment selection to preserve its role as an unbiased final evaluation dataset.

MLflow provides persistent evidence of the experiments, making it possible to compare configurations independently of notebook memory.

The selected candidate and its MLflow Run ID provide the handoff to the model-registration stage.

##### Next Notebook

##### 08_model_registration

The next notebook will take the selected neural-network candidate and move it into the controlled model lifecycle.

The workflow will be:

``` text

Selected Experiment
        ↓
MLflow Run
        ↓
Selected Model Artifact
        ↓
Model Validation
        ↓
Unity Catalog Model Registration
        ↓
Registered Model Version
        ↓
Model Alias / Lifecycle Management

```

The goal is to move from: 

"this experiment performed well"

to:

"this is a controlled, versioned model that can be referenced reliably by downstream production systems."